# 01 · 单场爬虫开发

## 这个 notebook 在做什么？

把 notebook 00 里调通的「调一次接口」逻辑，**封装成一个能稳定跑的函数**。

## 为什么要单独花一个 notebook 来做？

你之前 D 盘那两个项目（`KPL数据分析python版`、`pythonProject5`）的爬虫有几个老毛病：

| 毛病 | 后果 |
|------|------|
| 爬虫逻辑写死在主脚本里 | 想换接口/参数，要改一堆地方 |
| 没有重试 | 网络一抖就断，几百场比赛跑一半挂了 |
| 没保存原始 JSON | 挂了重头爬，被反爬封 IP |
| 没加延时 | 短时间高频请求，IP 被封 |
| `print` 满屏跑 | 跑完看不出哪些成功哪些失败 |

本 notebook 就是把这些坑**一个个补上**——而且**不是一次写完美**，而是 v1 → v2 → v3 → v4 **逐步迭代**。这是工程化思维：永远从最小可用版本开始。

## 跑完 notebook，你应该收获什么？

1. ✅ `fetch_battle(battle_id)` 函数：带重试 + 缓存 + 日志
2. ✅ 测试了 3 种异常场景（网络断、ID 不存在、JSON 解析失败）
3. ✅ 把最终代码迁移到 `src/crawler/api_client.py`，可以被其他 notebook `import`
4. ✅ `data/raw/` 文件夹下囤了几个真实的原始 JSON

## 工程哲学

> "Make it work, make it right, make it fast" —— Kent Beck
>
> 先让它能跑（v1）→ 再让它正确（v2 错误处理 / v3 重试）→ 最后让它高效（v4 缓存）。
> 一上来想完美 = 写不出来。

---
## 步骤 1 · 准备工作

### 思路

这次比 notebook 00 多用几个包：

- `time` / `random` —— 反爬延时（等待时间要随机，否则机器特征明显）
- `logging` —— **企业级别项目从来不用 `print` 调试，全部用 logger**
  - 好处 1：自带时间戳
  - 好处 2：可分级（INFO / WARNING / ERROR），调试时关 INFO，正式跑只看 ERROR
  - 好处 3：可同时输出到文件，跑完能复盘历史

### 任务

In [ ]:
# TODO 1.1：导包
# requests / json / time / random / logging / pathlib.Path
import requests
import json
import time
import random
import logging
from pathlib import Path


In [ ]:
# TODO 1.2：配置 logging
# 标准模板（可以直接抄但要看懂每一行）：
#
#   logging.basicConfig(
#       level=logging.INFO,
#       format='%(asctime)s | %(levelname)s | %(message)s',
#       datefmt='%H:%M:%S',
#   )
#   logger = logging.getLogger('crawler')
#
# 测试一下：logger.info('logger ok')，看输出是否带时间戳
logging.basicConfig(
	level = logging.INFO,
	format = '%(asctime)s | %(levelname)s | %(message)s',
	datefmt = '%H:%M:%S'
)

logger = logging.getLogger('crawler')

In [ ]:
logger.info('loogger ok')

15:42:48 | INFO | loogger ok


In [ ]:
# TODO 1.3：常量
# - BASE_URL（同 notebook 00）
# - HEADERS（同 notebook 00）
# - RAW_DIR = Path('../data/raw')
# - TIMEOUT = 15        # 秒，超过认为请求失败
# - MAX_RETRY = 3       # 重试次数
# - SLEEP_RANGE = (1.0, 2.0)  # 反爬延时区间，单位秒
#
# 思考：为什么延时要随机区间，不直接固定 1 秒？
# 提示：固定时间间隔的请求 = 机器特征 = 容易被反爬识别

BASE_URL = "https://prod.comp.smoba.qq.com/leaguesite/battle/open"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/128.0.0.0 Mobile Safari/537.36 Edg/128.0.0.0",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://pvp.qq.com/",
    "Origin": "https://pvp.qq.com",
}

RAW_DIR = Path('../data/raw').resolve()

TIMEOUT = 15
MAX_RETRY = 3
SLEEP_RANGE = (1.0,2.0)

_---
## 步骤 2 · v1：最简单的版本

### 思路

**别一上来就想完美**。先写最朴素能跑的版本，跑通后再加功能。

v1 的目标：传入一个 `battle_id`，返回解析后的字典。**不做任何错误处理。**

为什么先写这种"裸版"？因为：
1. 验证「URL 拼接 + Headers」是对的
2. 心里有底了，再加错误处理才知道在哪些地方加

In [ ]:
# TODO 2.1：写 fetch_battle_v1(battle_id)
# 函数体只做三件事：
#   1. 拼 URL
#   2. requests.get(url, headers=..., timeout=...)
#   3. return resp.json()
# 不要 try/except，不要重试，不要缓存

def fetch_battle_v1(battle_id):
    url = f"{BASE_URL}?battle_id={battle_id}"
    response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    return response.json()

# 测试：用 notebook 00 验证过的 battle_id 调一次
# 期望：拿到 dict，code == 0


In [ ]:
fetch_battle_v1('736117264_10_1777089898')

{'message': '',
 'code': 200,
 'request_id': '1ab9a8fdd7aa7145627a02e22ae6974b',
 'data': {'battle_id': '736117264_10_1777089898',
  'status': 2,
  'win_camp': 1,
  'game_duration': 835000,
  'battle_seq': 1,
  'camp1': {'team_id': '10017',
   'team_name': '广州TTG',
   'team_icon': 'https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823db5c8c44fae6abe.png',
   'is_win': True,
   'kill_num': 8,
   'assist_num': 22,
   'death_num': 4,
   'gold': 47042,
   'kill_big_dragon_num': 1,
   'push_tower_num': 5,
   'kill_dark_tyrant_num': 1,
   'kill_tyrant_num': 1,
   'kda': 7.5,
   'kill_prophet_dragon_num': 1,
   'kill_shadow_dragon_num': 1,
   'kill_storm_dragon_king_num': 0,
   'team_abbreviation': 'TTG'},
  'camp2': {'team_id': '10005',
   'team_name': 'KSG',
   'team_icon': 'https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89c5cf7e04d7f83534f.png',
   'is_win': False,
   'kill_num': 4,
   'assist_num': 9,
   'death_num': 8,
   'gold': 39591,
   'kill_big_dragon_num': 0,
   'push_tower_num': 2,
   'kill

💡 **思考一下**（写下来，不必长）：

- v1 跑通了吗？
- 如果给一个**故意错的** battle_id（比如 `xxx_999_yyy`），v1 会怎么样？返回 None？报异常？还是返回一个 code != 0 的 dict？
- v1 有哪些**显而易见的问题**？至少列 2 条

---
## 步骤 3 · v2：加错误处理

### 思路

爬数据会遇到的错误分两种：

**A. 技术性错误**（连接层面的）：
- `requests.exceptions.Timeout` —— 接口太慢
- `requests.exceptions.ConnectionError` —— 网断了 / DNS 解析失败
- HTTP 状态码不是 2xx —— 用 `resp.raise_for_status()` 触发
- `json.JSONDecodeError` —— 返回的不是合法 JSON

**B. 业务性错误**（接口能正常返回，但内容有问题）：
- `code != 0` —— 比如 ID 不存在，腾讯接口返回 `{"code": 1, "message": "..."}`
- 返回了 dict 但缺少 `data` 字段

**关键设计**：失败时返回 `None`，而不是抛异常。理由是后面要批量爬几百场，**单条失败不能让整个程序崩**。让调用方自己判断 `if data is None` 跳过即可。

In [ ]:
# TODO 3.1：写 fetch_battle_v2(battle_id)
# 在 v1 基础上加：
#   - 用 try/except 捕捉技术性错误（建议捕捉 requests.RequestException 这个父类，一网打尽）
#   - resp.raise_for_status() 检查 HTTP 状态
#   - 业务性检查：if data.get('code') != 0: 记日志返回 None
#   - 所有 print 改成 logger.xxx：
#       * logger.info()  —— 正常流程
#       * logger.warning() —— 业务异常但不致命（比如 code != 0）
#       * logger.error()  —— 技术性异常

def fetch_battle_v2(battle_id):
    try :
        url = f"{BASE_URL}?battle_id={battle_id}"
        response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        logger.error(e)
        return None
    if data.get('code')!=200:
	    logger.warning(data.get('code'))
	    return None
    elif data.get('data') is None:
	    logger.warning(data.get('data'))
	    return None
    logger.info("成功")
    return data


In [ ]:
# TODO 3.2：测试异常场景
# 至少试 2 种：
#   1. 故意改一个不存在的 battle_id（局数 = 999）
#   2. 故意把 BASE_URL 末尾改错（看连接错误是否被捕获）
# 期望：函数返回 None，且日志里有清晰的失败原因

In [ ]:
battle_id = '736117264_999_1777089898'
fetch_battle_v2(battle_id)

15:43:33 | WARNING | 404


In [ ]:
battle_id = '736117264_10_1777089898'
fetch_battle_v2(battle_id)

15:43:51 | INFO | 成功


{'message': '',
 'code': 200,
 'request_id': 'e0611e99c765637f858ef24881598bc6',
 'data': {'battle_id': '736117264_10_1777089898',
  'status': 2,
  'win_camp': 1,
  'game_duration': 835000,
  'battle_seq': 1,
  'camp1': {'team_id': '10017',
   'team_name': '广州TTG',
   'team_icon': 'https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823db5c8c44fae6abe.png',
   'is_win': True,
   'kill_num': 8,
   'assist_num': 22,
   'death_num': 4,
   'gold': 47042,
   'kill_big_dragon_num': 1,
   'push_tower_num': 5,
   'kill_dark_tyrant_num': 1,
   'kill_tyrant_num': 1,
   'kda': 7.5,
   'kill_prophet_dragon_num': 1,
   'kill_shadow_dragon_num': 1,
   'kill_storm_dragon_king_num': 0,
   'team_abbreviation': 'TTG'},
  'camp2': {'team_id': '10005',
   'team_name': 'KSG',
   'team_icon': 'https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89c5cf7e04d7f83534f.png',
   'is_win': False,
   'kill_num': 4,
   'assist_num': 9,
   'death_num': 8,
   'gold': 39591,
   'kill_big_dragon_num': 0,
   'push_tower_num': 2,
   'kill

💡 **观察**：

- v2 的日志输出长什么样？比 v1 的 print 更清晰吗？__清晰__
- 如果一个 battle_id 暂时网络抖动失败，v2 会怎么处理？（提示：不会重试，直接 return None）返回无
- 这意味着 v2 还有什么问题？重试等等解决方法

---
## 步骤 4 · v3：加重试（指数退避）

### 思路

网络请求一个朴素的真理：**第一次失败大概率第二次成功**。原因可能是临时拥塞、DNS 偶发抖动、对方限流暂时拒绝……

最简单的应对：失败了**等一会儿再试**。

但等多久？有讲究：

| 策略 | 做法 | 优劣 |
|------|------|------|
| 固定等待 | 每次都等 1 秒 | 简单但不友好——如果对方限流，你 1 秒后又来一次照样被拒 |
| 指数退避 | 第 N 次等 `2^N` 秒 | 越失败等越久，给对方"喘息"，业界标准 |
| 指数退避 + 抖动 | `2^N + random()` | 多个程序同时重试时不会撞车 |

我们用第三种。

In [ ]:
# TODO 4.1：写 fetch_battle_v3(battle_id, max_retry=3)
# 框架：
#   for attempt in range(max_retry):
#       try:
#           ... 请求逻辑（v2 的核心部分）...
#           return data
#       except (Timeout, ConnectionError) as e:
#           wait = 2 ** attempt + random.uniform(0, 1)
#           logger.warning(f'第 {attempt+1} 次失败，{wait:.1f}s 后重试：{e}')
#           time.sleep(wait)
#   logger.error(f'{battle_id} 重试 {max_retry} 次仍失败')
#   return None
#
# 思考：
#   - 业务错误（code != 200）要不要重试？为什么？
#     提示：不要！业务错误重试 100 次也是错的，浪费时间

def fetch_battle_v3(battle_id, max_retry=3):
	for i in range(max_retry):
		try :
			url = f"{BASE_URL}?battle_id={battle_id}"
			response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
			response.raise_for_status()
			data = response.json()
			if data.get('code')!=200:
				logger.warning(data.get('code'))
				return None
			elif data.get('data') is None:
				logger.warning(data.get('data'))
				return None
			logger.info(f"[正常] 成功获取数据, battle_id: {battle_id}")
			return data
		except (requests.exceptions.Timeout,requests.exceptions.ConnectionError) as e:
			logger.warning(e)
			wait_time = 2 ** i + random.uniform(0, 1)
			logging.warning(f"第{i+1}次失败，等待时间：{wait_time}，后重试：{e}")
			time.sleep(wait_time)
	logger.error(f"{battle_id},已经重试{max_retry}次均失败")
	return None


In [ ]:
# TODO 4.2：测试重试逻辑
# 把 BASE_URL 改成一个不存在的域名（比如 https://thisurldoesnotexist.qq.com/...）
# 调一次 fetch_battle_v3，看日志里：
#   - 是不是真的重试了 3 次？
#   - 每次等待时间是 1s、2s、4s（左右）吗？
# 测试完记得把 BASE_URL 改回来！
fetch_battle_v3("736117264_10_1777089898")


15:43:54 | INFO | [正常] 成功获取数据, battle_id: 736117264_10_1777089898


{'message': '',
 'code': 200,
 'request_id': '06ded9b9a06ec572ac6fbeb8316eaa76',
 'data': {'battle_id': '736117264_10_1777089898',
  'status': 2,
  'win_camp': 1,
  'game_duration': 835000,
  'battle_seq': 1,
  'camp1': {'team_id': '10017',
   'team_name': '广州TTG',
   'team_icon': 'https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823db5c8c44fae6abe.png',
   'is_win': True,
   'kill_num': 8,
   'assist_num': 22,
   'death_num': 4,
   'gold': 47042,
   'kill_big_dragon_num': 1,
   'push_tower_num': 5,
   'kill_dark_tyrant_num': 1,
   'kill_tyrant_num': 1,
   'kda': 7.5,
   'kill_prophet_dragon_num': 1,
   'kill_shadow_dragon_num': 1,
   'kill_storm_dragon_king_num': 0,
   'team_abbreviation': 'TTG'},
  'camp2': {'team_id': '10005',
   'team_name': 'KSG',
   'team_icon': 'https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89c5cf7e04d7f83534f.png',
   'is_win': False,
   'kill_num': 4,
   'assist_num': 9,
   'death_num': 8,
   'gold': 39591,
   'kill_big_dragon_num': 0,
   'push_tower_num': 2,
   'kill

---
## 步骤 5 · v4：加缓存 ⭐⭐⭐

### 思路

**这一步最值钱**。直接把爬虫效率提升 100 倍。

观察一个事实：**赛后比赛的数据是固定不变的**——`battle_id = 1038107152_18_1742644777` 这场比赛的 JSON，今天爬和明天爬完全一样。

那爬过的就别再爬了——**直接读本地文件**。

### 缓存策略

```
fetch_battle('xxx')
  ├── 检查 data/raw/xxx.json 是否存在
  │     ├── 存在 → 读本地文件返回（耗时 < 1ms）
  │     └── 不存在 → 走网络请求（耗时 200ms+），成功后保存到 data/raw/xxx.json
  └── 返回 dict
```

### 好处

1. **调试时不重复请求**——防被反爬
2. **中途挂了不重头来**——已经爬过的下次跳过
3. **本地秒回**——开发体验直线上升

### 注意

> 这个策略对**赛后数据**有效。**实时数据**（V2 阶段）规则不一样，因为同一个 battle_id 在比赛进行中数据会变。先把赛后版本做好，V2 阶段再升级。

In [ ]:
# TODO 5.1：写 fetch_battle(battle_id, use_cache=True) —— 最终版
#
# 完整流程：
#   1. cache_path = RAW_DIR / f'{battle_id}.json'
#   2. if use_cache and cache_path.exists():
#         logger.info(f'[缓存命中] {battle_id}')
#         with open(cache_path, encoding='utf-8') as f:
#             return json.load(f)
#   3. 走 v3 的重试逻辑，拿到 data
#   4. 如果 data is not None：保存到 cache_path
#         with open(cache_path, 'w', encoding='utf-8') as f:
#             json.dump(data, f, ensure_ascii=False, indent=2)
#   5. time.sleep(random.uniform(*SLEEP_RANGE))   # 反爬延时（注意：缓存命中时不该 sleep）
#   6. return data
#
# 思考：步骤 5 的 sleep 应该放在 if 之外还是之内？
# 提示：放在 "网络请求" 分支里，缓存命中时跳过 sleep（不然每次都白等）

def fetch_battle_V4(battle_id, use_cache=True):

    cache_path = RAW_DIR / f'{battle_id}.json'

    if use_cache and cache_path.exists():

        logger.info(f"[命中缓存]{battle_id}")

        with open(cache_path, encoding='utf-8-sig') as f:
            return json.load(f)

    else:

        for i in range(MAX_RETRY):

            try:

                url = f"{BASE_URL}?battle_id={battle_id}"

                response = requests.get(
                    url,
                    headers=HEADERS,
                    timeout=TIMEOUT
                )

                response.raise_for_status()

                data = response.json()

                if data.get('code') != 200:

                    logger.warning(data.get('code'))
                    return None

                elif data['data'] is None:

                    logger.warning(data.get('data'))
                    return None

                logger.info(f'[网络请求]:{battle_id},已成功。')

                if data is not None:

                    with open(cache_path,'w', encoding='utf-8-sig') as f:

                        json.dump(
                            data,
                            f,
                            ensure_ascii=False,
                            indent=2
                        )

                    time.sleep(
                        random.uniform(
                            SLEEP_RANGE[0],
                            SLEEP_RANGE[1]
                        )
                    )

                return data

            except (
                requests.exceptions.Timeout,
                requests.exceptions.ConnectionError
            ) as e:

                logger.warning(e)

                wait_time = (
                    2 ** i + random.uniform(0, 1)
                )

                logger.warning(
                    f"第{i+1}次失败，等待时间："
                    f"{wait_time}，后重试：{e}"
                )

                time.sleep(wait_time)

        logger.error(f"{battle_id},已经重试{MAX_RETRY}次均失败")

        return None

In [ ]:
# TODO 5.2：测试缓存
# 1. 第一次调 fetch_battle('xxx')：日志应该说"网络请求成功"，并 sleep 1-2 秒
# 2. 第二次调同一个 ID：日志说"[缓存命中]"，秒回（< 0.01 秒）
# 3. 用 time.perf_counter() 计时对比，体感一下差别
# 4. 去 data/raw/ 文件夹看看有没有新文件生成

import time
t1 = time.perf_counter()
# 第一次调用
fetch_battle_V4('736117264_10_1777089898')
t2 = time.perf_counter()
# 第二次调用
fetch_battle_V4("736117264_10_1777089898")
t3 = time.perf_counter()
# 打印两次耗时
print(t2-t1)
print(t3-t2)

15:43:57 | INFO | [命中缓存]736117264_10_1777089898
15:43:57 | INFO | [命中缓存]736117264_10_1777089898


0.002449799794703722
0.0017091999761760235


💡 **观察**：

- 网络请求耗时多少？缓存命中耗时多少？倍数是？
- 这个差距对批量爬全季 100+ 场比赛意味着什么？（提示：调试期反复跑就不再"花钱"了）

---
## 步骤 6 · 小批量实战

### 思路

理论再完美也得跑一次实战。用 notebook 00 验证过的编码规则，构造同一场 BO 系列的多个 battle_id，批量爬一下。

这一步的意义：
- 验证函数在**连续多次调用**下稳定吗？
- 反爬延时配置合理吗？（如果 1-2 秒还被封，要拉长到 3-5 秒）
- 数据真的存到 `data/raw/` 了吗？

In [ ]:
# TODO 6.1：构造 5 个 battle_id 批量爬
# 用 notebook 00 步骤 7 验证过的编码规则，比如：
#   ids = [f'1038107152_{n}_1742644777' for n in range(18, 23)]
#
# 用 tqdm 显示进度条：
#   from tqdm.auto import tqdm
#   for bid in tqdm(ids):
#       data = fetch_battle(bid)
#
# 收集成功 / 失败的列表



In [ ]:
MATCH_URL = "https://prod.comp.smoba.qq.com/leaguesite/match/battles/open?"
match_id = '2026042502'
match_url = f'{MATCH_URL}match_id={match_id}'
response = requests.get(match_url, headers=HEADERS, timeout=TIMEOUT)
response.raise_for_status()
data_match = response.json()

In [83]:
from tqdm import tqdm
ids = []
for result in data_match['results']:
	logger.info(f"成功获取{match_id}当天赛场内的比赛id")
	ids.append(result['battle_id'])

18:49:51 | INFO | 成功获取2026042502当天赛场内的比赛id
18:49:51 | INFO | 成功获取2026042502当天赛场内的比赛id
18:49:51 | INFO | 成功获取2026042502当天赛场内的比赛id
18:49:51 | INFO | 成功获取2026042502当天赛场内的比赛id


In [86]:
ids

['736117264_14_1777099889',
 '736117264_15_1777101864',
 '736117264_16_1777103765',
 '736117264_17_1777105595']

In [71]:
# TODO 6.2：检查产出
# 1. 成功几条？失败几条？
# 2. data/raw/ 文件夹下文件数对不上吗？
#    提示：用 list(RAW_DIR.glob('*.json')) 数文件
# 3. 随便打开一个 .json 文件看看里面有没有正确的中文（不是 \uXXXX）

len(list(RAW_DIR.glob('*.json')))

5

💡 **观察**：

- 5 条全成功了吗成功
- 总耗时大概多少秒？（应该 = 网络耗时 × 5 + 延时 × 5 ≈ 6-15 秒）
- 再跑一次（同样的 ids），耗时变化大吗？为什么？

---
## 步骤 7 · 沉淀到 src 模块（工程化）

### 思路

**Notebook 是实验场，src 是产品**。

调试通过的代码不能一直留在 notebook 里——别的 notebook 想用这个函数怎么办？复制粘贴吗？那 bug 修一处要改 N 处。

正确做法：把最终版 `fetch_battle` 函数迁移到 `src/crawler/api_client.py`，常量迁移到 `src/config.py`。然后所有 notebook `from src.crawler.api_client import fetch_battle`，整个项目共用一份代码。

### 任务

1. 打开 `src/config.py`，把 `BASE_URL_BATTLE` 和 `HEADERS` 填上（之前是空字符串）
2. 打开 `src/crawler/api_client.py`，把 `fetch_battle` 函数搬过去
   - 顶部加：`from src.config import BASE_URL_BATTLE, HEADERS, RAW_DIR, TIMEOUT, MAX_RETRY, SLEEP_RANGE`
   - 顶部加：`from src.utils.logger import get_logger; logger = get_logger(__name__)`
   - 函数加完整的 docstring（参数说明 / 返回值 / 异常）
   - 加类型注解：`def fetch_battle(battle_id: str, use_cache: bool = True) -> dict | None:`
3. 回到这个 notebook 验证 import 能跑通

In [72]:
# TODO 7.1：从 src 导入并验证
# 注意：notebook 在 notebooks/ 文件夹，要把项目根目录加到 sys.path

import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

# from src.crawler.api_client import fetch_battle
# data = fetch_battle('1038107152_18_1742644777')
# print(data['data']['battle_id'])



In [74]:
print(sys.path.insert(0, str(Path('..').resolve())))

None


In [87]:
from src.crawler.api_client import  fetch_battle
for bid in tqdm(ids):
    fetch_battle(bid)

  0%|          | 0/4 [00:00<?, ?it/s]

2026-05-07 18:53:03 | src.crawler.api_client | INFO | 已查询到数据，正在写入数据


18:53:03 | INFO | 已查询到数据，正在写入数据


2026-05-07 18:53:05 | src.crawler.api_client | INFO | 数据保存成功，路径：D:\AI数据分析\项目作品集\02-KPL实时胜率预测系统\data\raw\736117264_14_1777099889.json


18:53:05 | INFO | 数据保存成功，路径：D:\AI数据分析\项目作品集\02-KPL实时胜率预测系统\data\raw\736117264_14_1777099889.json
 25%|██▌       | 1/4 [00:02<00:07,  2.49s/it]

...

18:53:08 | INFO | 已查询到数据，正在写入数据


2026-05-07 18:53:09 | src.crawler.api_client | INFO | 数据保存成功，路径：D:\AI数据分析\项目作品集\02-KPL实时胜率预测系统\data\raw\736117264_16_1777103765.json


18:53:09 | INFO | 数据保存成功，路径：D:\AI数据分析\项目作品集\02-KPL实时胜率预测系统\data\raw\736117264_16_1777103765.json
 75%|███████▌  | 3/4 [00:06<00:02,  2.09s/it]

2026-05-07 18:53:09 | src.crawler.api_client | INFO | [命中缓存，736117264_17_1777105595]


18:53:09 | INFO | [命中缓存，736117264_17_1777105595]
100%|██████████| 4/4 [00:06<00:00,  1.63s/it]


---
## ✅ 完成自检

- [ ] v1（最简版）能跑通
- [ ] v2 加了错误处理，3 种异常都能优雅返回 None
- [ ] v3 加了重试，亲眼看到日志里指数退避
- [ ] v4 加了缓存，第二次调用秒回
- [ ] 小批量爬了至少 3 场比赛，全部入 `data/raw/`
- [ ] 代码已经迁移到 `src/crawler/api_client.py`
- [ ] `src/config.py` 的 `BASE_URL_BATTLE` 和 `HEADERS` 已填好
- [ ] notebook 末尾从 `src.crawler.api_client` import 验证成功

## 🎯 完成后跟我汇报

```
1. 我做了什么：______
2. 我的思考：______（重点：v1 → v4 每次升级解决了什么问题？）
3. 我想确认：______
```

me:
- V1版本是最小模块，我 将最基础的爬取放入函数之中；V2版本下加入错误管理，利用try和except，使得发生异常时也能返回None;V3版本我对超时异常和连接异常的情况加入重试；V4版本则加入缓存机制，节省时间；另外在V4版本，我考虑到，其他异常没有加入错误处理，于是自己加入父类错误，利用expect的顺序，来实现全部错误拦截。学会了怎么从另一个脚本里面导入对应的函数，通过from目录import函数名，来实现。
- 我想知道函数调用的脚本里面，我需要导入库吗？如果我只是把函数赋值，而常量什么的没有搬过去，我在调用的时候也能运行成功吗？为此我实验了一下，发现：from src.config import (
    BASE_URL_BATTLE,
    HEADERS,
    RAW_DIR,
    TIMEOUT,
    MAX_RETRY,
    SLEEP_RANGE
)是可以不用放入api_client脚本里面的。
-其次我在脚本利用我在00里面的发现，利用赛程url爬取match_id，在利用这个，获取到一场比赛里面的所有对局的情况。接下来我会验证一天中有几场比赛情况，链接里面是否是包含所有battle_id。
-结果：存在，且只需要在第一个match_id+1就等于第二次的match_id。以上结论均已验证。